Cellule 1 : GPU, Drive, COLMAP, dépôt

In [ ]:
import os, torch
print("GPU :", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
!apt-get install -y -qq colmap > /dev/null 2>&1
!colmap -h | head -n 2
if not os.path.exists("/content/gaussian-splatting"):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting
VIDEO = "/content/drive/MyDrive/video_panda.mov"
print("Vidéo trouvée :", os.path.exists(VIDEO))

Cellule 2 : compilation CUDA (3 à 6 min)

In [ ]:
from pathlib import Path
h = Path("/content/gaussian-splatting/submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h")
t = h.read_text()
if "#include <cstdint>" not in t:
    h.write_text("#include <cstdint>\n#include <cstddef>\n#include <stdint.h>\n" + t)
%cd /content/gaussian-splatting
!pip install --no-build-isolation --no-cache-dir ./submodules/diff-gaussian-rasterization 2>&1 | tail -n 2
!pip install --no-build-isolation --no-cache-dir ./submodules/simple-knn 2>&1 | tail -n 2
!pip install -q plyfile lpips
import diff_gaussian_rasterization, simple_knn
print("OK")

Cellule 3 : extraire les images de la vidéo

In [ ]:
import glob, shutil, cv2, numpy as np
from PIL import Image
shutil.rmtree("/content/frames", ignore_errors=True); os.makedirs("/content/frames")
!ffmpeg -loglevel error -y -i "{VIDEO}" -vf "fps=3,scale=720:-2" -qscale:v 2 /content/frames/f_%04d.jpg

files = sorted(glob.glob("/content/frames/*.jpg"))
score = [cv2.Laplacian(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2GRAY), cv2.CV_64F).var() for f in files]
keep = [f for f, s in zip(files, score) if s >= np.percentile(score, 15)]   # on retire les 15 % les plus flous
MAX = 90
if len(keep) > MAX:
    keep = [keep[i] for i in np.linspace(0, len(keep) - 1, MAX).astype(int)]
shutil.rmtree("/content/scene3", ignore_errors=True); os.makedirs("/content/scene3/input")
for i, f in enumerate(keep):
    shutil.copy(f, f"/content/scene3/input/img_{i:03d}.jpg")
print(len(files), "images extraites ->", len(keep), "gardées")
print("Taille d'une image (largeur, hauteur) :", Image.open("/content/scene3/input/img_000.jpg").size)

Cellule 4 : COLMAP, extraction des points (environ 3 min)

In [ ]:
!mkdir -p /content/scene3/distorted/sparse
!colmap feature_extractor --database_path /content/scene3/distorted/database.db --image_path /content/scene3/input --ImageReader.single_camera 1 --ImageReader.camera_model SIMPLE_RADIAL --SiftExtraction.use_gpu 0 --SiftExtraction.max_num_features 4000 2>&1 | tail -n 3

Cellule 5 : COLMAP, appariement (15 à 25 min, la plus longue)

In [ ]:
!colmap exhaustive_matcher --database_path /content/scene3/distorted/database.db --SiftMatching.use_gpu 0 2>&1 | tail -n 3

Cellule 6 : COLMAP, poses de caméras et point de contrôle

In [ ]:
!colmap mapper --database_path /content/scene3/distorted/database.db --image_path /content/scene3/input --output_path /content/scene3/distorted/sparse 2>&1 | tail -n 3
print("Sous-dossiers de sparse :")
!ls /content/scene3/distorted/sparse
!colmap model_analyzer --path /content/scene3/distorted/sparse/0 2>&1 | grep -E "Cameras|Registered|Points:|reprojection"

Cellule 7 : conversion pour le Gaussian Splatting et sauvegarde

In [ ]:
!cd /content/gaussian-splatting && python convert.py -s /content/scene3 --no_gpu --skip_matching 2>&1 | tail -n 2
!ls /content/scene3/images | wc -l
!ls /content/scene3/sparse/0
!cp -r /content/scene3 /content/drive/MyDrive/scene_gs_v3
print("Scène sauvegardée sur Drive")

Cellule 8 : diagnostic des caméras

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "/content/gaussian-splatting")
from scene.colmap_loader import read_extrinsics_binary, qvec2rotmat

ext = read_extrinsics_binary("/content/scene3/sparse/0/images.bin")
items = sorted(ext.values(), key=lambda im: im.name)
C, D, names = [], [], []
for im in items:
    R = qvec2rotmat(im.qvec)
    C.append(-R.T @ im.tvec); D.append(R.T @ np.array([0, 0, 1.0])); names.append(im.name)
C, D = np.array(C), np.array(D)
test = np.array([i % 8 == 0 for i in range(len(items))])

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(121, projection="3d")
ax.scatter(*C[~test].T, c="tab:blue", s=15, label="train")
ax.scatter(*C[test].T, c="tab:red", s=40, label="test"); ax.legend(); ax.set_title("Positions des caméras")
ax2 = fig.add_subplot(122)
ax2.scatter(C[~test, 0], C[~test, 2], c="tab:blue", s=15); ax2.scatter(C[test, 0], C[test, 2], c="tab:red", s=40)
ax2.set_aspect("equal"); ax2.set_title("Vue de dessus (x, z)")
plt.show()

angles = []
for i in np.where(test)[0]:
    cos = (D[~test] @ D[i]).max()
    angles.append(np.degrees(np.arccos(np.clip(cos, -1, 1))))
print("Angle jusqu'à la vue d'entraînement la plus proche (test) :", [round(a, 1) for a in angles])

Cellule 9 : entraînement de validation (7 000 itérations, environ 10 min)

In [ ]:
import time; t0 = time.time()
!cd /content/gaussian-splatting && python train.py -s /content/scene3 -m /content/output/panda_v3 --eval --iterations 7000 --test_iterations 7000 --save_iterations 7000 2>&1 | tail -n 8
print("Temps d'entraînement (s) :", round(time.time() - t0))
!cp -r /content/output/panda_v3 /content/drive/MyDrive/panda_gs_v3

Cellule 10 : rendu, métriques et figure

In [ ]:
!cd /content/gaussian-splatting && python render.py -m /content/output/panda_v3 --iteration 7000 --skip_train 2>&1 | tail -n 2
!cd /content/gaussian-splatting && python metrics.py -m /content/output/panda_v3 2>&1 | tail -n 6
import glob
b = "/content/output/panda_v3/test/ours_7000"
R = sorted(glob.glob(f"{b}/renders/*.png"))[:3]; G = sorted(glob.glob(f"{b}/gt/*.png"))[:3]
fig, ax = plt.subplots(2, len(R), figsize=(4 * len(R), 7))
for i in range(len(R)):
    ax[0, i].imshow(Image.open(G[i])); ax[0, i].axis("off")
    ax[1, i].imshow(Image.open(R[i])); ax[1, i].axis("off")
fig.suptitle("Haut : photos réelles (test) / Bas : rendus")
plt.show()